# Jump-Diffusion Model (Merton, 1976)

## Motivation

The Black-Scholes model assumes asset prices follow a continuous geometric Brownian motion. In reality, prices exhibit **sudden jumps** — earnings surprises, crashes, or news events — that pure diffusion cannot capture. This leads to:

- **Leptokurtic returns** (fat tails): extreme moves are more frequent than GBM predicts
- **Volatility smile**: implied volatility varies with strike, which BS can't explain

Merton's jump-diffusion model augments GBM with a **compound Poisson process** to capture these discontinuities.

## The Model

The asset price follows:

$$\frac{dS}{S} = (\mu - \lambda \bar{k})\,dt + \sigma\,dW_t + (e^J - 1)\,dN_t$$

where:
- $W_t$: standard Wiener process (continuous diffusion)
- $N_t$: Poisson process with intensity $\lambda$ (jump arrivals)
- $J \sim N(\mu_J, \sigma_J^2)$: log-jump size (normally distributed)
- $\bar{k} = \mathbb{E}[e^J - 1] = e^{\mu_J + \sigma_J^2/2} - 1$: compensator ensuring $\mathbb{E}[dS/S] = \mu\,dt$

Under the **risk-neutral** measure, the exact solution is:

$$S_T = S_0 \exp\!\left[\left(r - \lambda\bar{k} - \frac{\sigma^2}{2}\right)T + \sigma W_T + \sum_{i=1}^{N_T} J_i \right]$$

where $N_T \sim \text{Poisson}(\lambda T)$ is the number of jumps in $[0, T]$, and $J_i \stackrel{iid}{\sim} N(\mu_J, \sigma_J^2)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, log, sqrt, exp, factorial
from scipy.stats import kurtosis, skew
%matplotlib inline

def bs_cdf(x):
    return 0.5 * (1 + erf(x / sqrt(2)))

def bs_call(S, K, r, T, sigma):
    d1 = (log(S/K) + (r + sigma**2/2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return S * bs_cdf(d1) - K * exp(-r*T) * bs_cdf(d2)

def bs_put(S, K, r, T, sigma):
    d1 = (log(S/K) + (r + sigma**2/2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return -S * bs_cdf(-d1) + K * exp(-r*T) * bs_cdf(-d2)

## Step 1: Simulate Jump-Diffusion Paths

We simulate full paths step-by-step. At each time step $\Delta t$:

$$S_{t+\Delta t} = S_t \exp\!\left[\left(r - \lambda\bar{k} - \frac{\sigma^2}{2}\right)\Delta t + \sigma\sqrt{\Delta t}\,Z + \sum_{i=1}^{n} J_i\right]$$

where $Z \sim N(0,1)$ and $n \sim \text{Poisson}(\lambda \Delta t)$.

In [ ]:
def simulate_jump_diffusion_paths(S0, r, sigma, T, lam, mu_J, sigma_J, n_steps, n_paths):
    """
    Simulate Merton jump-diffusion paths under the risk-neutral measure.
    
    S0: initial price
    r: risk-free rate
    sigma: diffusion volatility
    T: time to maturity
    lam: jump intensity (expected number of jumps per year)
    mu_J: mean of log-jump size
    sigma_J: std of log-jump size
    n_steps: number of time steps
    n_paths: number of MC paths
    
    Returns: array of shape (n_steps+1, n_paths)
    """
    dt = T / n_steps
    k_bar = np.exp(mu_J + 0.5 * sigma_J**2) - 1  # compensator
    
    # Pre-allocate log-price array
    log_S = np.zeros((n_steps + 1, n_paths))
    log_S[0] = np.log(S0)
    
    drift = (r - lam * k_bar - 0.5 * sigma**2) * dt
    
    for t in range(n_steps):
        # Diffusion component
        Z = np.random.randn(n_paths)
        diffusion = sigma * np.sqrt(dt) * Z
        
        # Jump component: number of jumps is Poisson(lam * dt)
        n_jumps = np.random.poisson(lam * dt, n_paths)
        # Sum of jump sizes: sum of n_jumps normal random variables
        # If n_jumps[i] = 0, jump_sum[i] = 0
        jump_sum = np.zeros(n_paths)
        has_jumps = n_jumps > 0
        if np.any(has_jumps):
            for i in np.where(has_jumps)[0]:
                jump_sum[i] = np.sum(np.random.normal(mu_J, sigma_J, n_jumps[i]))
        
        log_S[t + 1] = log_S[t] + drift + diffusion + jump_sum
    
    return np.exp(log_S)


def simulate_jump_diffusion_terminal(S0, r, sigma, T, lam, mu_J, sigma_J, n_paths):
    """
    Simulate only the terminal price S_T (exact, no time-stepping error).
    Faster than full path simulation when only S_T is needed.
    """
    k_bar = np.exp(mu_J + 0.5 * sigma_J**2) - 1
    
    # Number of jumps over [0, T]
    N = np.random.poisson(lam * T, n_paths)
    
    # Sum of jump sizes
    # For each path, sum N[i] normal random variables
    jump_sum = np.zeros(n_paths)
    max_jumps = N.max() if N.max() > 0 else 0
    for j in range(1, max_jumps + 1):
        mask = N >= j
        jump_sum[mask] += np.random.normal(mu_J, sigma_J, np.sum(mask))
    
    Z = np.random.randn(n_paths)
    log_S_T = (np.log(S0) 
               + (r - lam * k_bar - 0.5 * sigma**2) * T 
               + sigma * np.sqrt(T) * Z 
               + jump_sum)
    return np.exp(log_S_T)

## Step 2: Visualize Jump-Diffusion vs Pure GBM

Let's compare sample paths to see how jumps create sudden discontinuities.

In [ ]:
np.random.seed(42)

# Parameters
S0, K, r, T, sigma = 100, 100, 0.05, 1.0, 0.15
lam = 3.0        # ~3 jumps per year
mu_J = -0.05     # jumps are slightly negative on average (crash-like)
sigma_J = 0.1    # jump size volatility
n_steps = 252

# Simulate paths
S_jd = simulate_jump_diffusion_paths(S0, r, sigma, T, lam, mu_J, sigma_J, n_steps, 10000)

# Pure GBM paths for comparison (same diffusion sigma)
Z_gbm = np.random.randn(n_steps, 10000)
log_inc_gbm = (r - 0.5*sigma**2) * (T/n_steps) + sigma * np.sqrt(T/n_steps) * Z_gbm
S_gbm = np.exp(np.vstack([np.full((1, 10000), np.log(S0)),
                           np.log(S0) + np.cumsum(log_inc_gbm, axis=0)]))

t_grid = np.linspace(0, T, n_steps + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for i in range(8):
    ax1.plot(t_grid, S_gbm[:, i], alpha=0.6, linewidth=0.8)
ax1.set_xlabel('Time')
ax1.set_ylabel('$S_t$')
ax1.set_title('Pure GBM Paths')
ax1.axhline(S0, color='k', linestyle='--', alpha=0.3)
ax1.set_ylim(40, 200)
ax1.grid(alpha=0.3)

for i in range(8):
    ax2.plot(t_grid, S_jd[:, i], alpha=0.6, linewidth=0.8)
ax2.set_xlabel('Time')
ax2.set_ylabel('$S_t$')
ax2.set_title(r'Jump-Diffusion Paths ($\lambda=3, \mu_J=-0.05$)')
ax2.axhline(S0, color='k', linestyle='--', alpha=0.3)
ax2.set_ylim(40, 200)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 3: Return Distribution — Fat Tails

A key signature of jump-diffusion is **fat tails** in the return distribution. Let's compare log-returns.

In [ ]:
log_ret_gbm = np.log(S_gbm[-1] / S0)
log_ret_jd = np.log(S_jd[-1] / S0)

fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(-1.0, 0.8, 150)
ax.hist(log_ret_gbm, bins=bins, density=True, alpha=0.5, label='GBM')
ax.hist(log_ret_jd, bins=bins, density=True, alpha=0.5, label='Jump-Diffusion')
ax.set_xlabel(r'Log-return $\ln(S_T/S_0)$')
ax.set_ylabel('Density')
ax.set_title('Return Distribution: Jump-Diffusion Has Fatter Tails')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("=== Return Distribution Statistics ===")
print(f"{'':15s} {'Mean':>10s} {'Std':>10s} {'Skewness':>10s} {'Kurtosis':>10s}")
print(f"{'GBM':15s} {np.mean(log_ret_gbm):10.4f} {np.std(log_ret_gbm):10.4f} "
      f"{skew(log_ret_gbm):10.4f} {kurtosis(log_ret_gbm):10.4f}")
print(f"{'Jump-Diffusion':15s} {np.mean(log_ret_jd):10.4f} {np.std(log_ret_jd):10.4f} "
      f"{skew(log_ret_jd):10.4f} {kurtosis(log_ret_jd):10.4f}")
print(f"\nGBM kurtosis ~ 0 (Gaussian). Jump-diffusion has excess kurtosis (fat tails)"
      f"\nand negative skewness (crash-like jumps with mu_J < 0).")

## Step 4: Merton's Analytical Formula

Merton showed that the jump-diffusion call price is a **Poisson-weighted sum of Black-Scholes prices**:

$$C_{\text{Merton}} = \sum_{n=0}^{\infty} \frac{e^{-\lambda' T}(\lambda' T)^n}{n!} \, C_{\text{BS}}(S_0, K, r_n, T, \sigma_n)$$

where:
- $\lambda' = \lambda(1 + \bar{k})$
- $r_n = r - \lambda\bar{k} + \frac{n \ln(1+\bar{k})}{T}$
- $\sigma_n = \sqrt{\sigma^2 + \frac{n \sigma_J^2}{T}}$

Each term corresponds to exactly $n$ jumps occurring during $[0, T]$.

In [ ]:
def merton_call(S0, K, r, T, sigma, lam, mu_J, sigma_J, n_terms=50):
    """Merton jump-diffusion call price (analytical)."""
    k_bar = exp(mu_J + 0.5 * sigma_J**2) - 1
    lam_prime = lam * (1 + k_bar)
    
    price = 0.0
    for n in range(n_terms):
        sigma_n = sqrt(sigma**2 + n * sigma_J**2 / T)
        r_n = r - lam * k_bar + n * log(1 + k_bar) / T
        # Poisson weight
        w = exp(-lam_prime * T) * (lam_prime * T)**n / factorial(n)
        price += w * bs_call(S0, K, r_n, T, sigma_n)
    return price

def merton_put(S0, K, r, T, sigma, lam, mu_J, sigma_J, n_terms=50):
    """Merton jump-diffusion put price (analytical)."""
    k_bar = exp(mu_J + 0.5 * sigma_J**2) - 1
    lam_prime = lam * (1 + k_bar)
    
    price = 0.0
    for n in range(n_terms):
        sigma_n = sqrt(sigma**2 + n * sigma_J**2 / T)
        r_n = r - lam * k_bar + n * log(1 + k_bar) / T
        w = exp(-lam_prime * T) * (lam_prime * T)**n / factorial(n)
        price += w * bs_put(S0, K, r_n, T, sigma_n)
    return price

## Step 5: Verify MC Against Analytical Formula

In [ ]:
# MC pricing using exact terminal simulation (no time-stepping error)
n_mc = 500000
S_T_jd = simulate_jump_diffusion_terminal(S0, r, sigma, T, lam, mu_J, sigma_J, n_mc)

call_payoff = np.maximum(S_T_jd - K, 0)
put_payoff = np.maximum(K - S_T_jd, 0)

mc_call = np.mean(np.exp(-r*T) * call_payoff)
mc_put = np.mean(np.exp(-r*T) * put_payoff)
mc_call_err = np.std(np.exp(-r*T) * call_payoff) / np.sqrt(n_mc)
mc_put_err = np.std(np.exp(-r*T) * put_payoff) / np.sqrt(n_mc)

# Analytical Merton prices
merton_c = merton_call(S0, K, r, T, sigma, lam, mu_J, sigma_J)
merton_p = merton_put(S0, K, r, T, sigma, lam, mu_J, sigma_J)

# BS prices (no jumps) for reference
bs_c = bs_call(S0, K, r, T, sigma)
bs_p = bs_put(S0, K, r, T, sigma)

print("=== Option Prices: BS vs Merton vs MC ===")
print(f"{'':10s} {'BS (no jumps)':>14s} {'Merton (analyt)':>16s} {'MC (500k paths)':>16s} {'MC Std Err':>10s}")
print(f"{'Call':10s} {bs_c:14.6f} {merton_c:16.6f} {mc_call:16.6f} {mc_call_err:10.6f}")
print(f"{'Put':10s} {bs_p:14.6f} {merton_p:16.6f} {mc_put:16.6f} {mc_put_err:10.6f}")
print(f"\nJumps increase option prices (especially puts) due to added tail risk.")

## Step 6: The Volatility Smile

If we price options under jump-diffusion and then invert each price using the BS formula (as a market participant would), we recover an **implied volatility** that varies with strike — the famous **volatility smile**.

In [ ]:
def implied_vol_newton(market_price, S, K, r, T, is_call=True, tol=1e-8, max_iter=100):
    """Find implied vol via Newton-Raphson."""
    sig = 0.3
    for _ in range(max_iter):
        if is_call:
            price = bs_call(S, K, r, T, sig)
        else:
            price = bs_put(S, K, r, T, sig)
        
        d1 = (log(S/K) + (r + sig**2/2)*T) / (sig*sqrt(T))
        vega = S * np.exp(-d1**2/2) / sqrt(2*np.pi) * sqrt(T)
        
        diff = price - market_price
        if abs(diff) < tol:
            return sig
        if vega < 1e-12:
            return np.nan
        sig -= diff / vega
        if sig < 1e-6:
            sig = 1e-6
    return np.nan

# Compute Merton prices across strikes, then invert to get implied vol
strikes = np.linspace(70, 135, 40)
iv_merton = []
iv_bs_flat = []

for Ki in strikes:
    # Merton price
    if Ki < S0:
        # Use put for OTM puts (better numerical stability)
        mp = merton_put(S0, Ki, r, T, sigma, lam, mu_J, sigma_J)
        iv = implied_vol_newton(mp, S0, Ki, r, T, is_call=False)
    else:
        mc = merton_call(S0, Ki, r, T, sigma, lam, mu_J, sigma_J)
        iv = implied_vol_newton(mc, S0, Ki, r, T, is_call=True)
    iv_merton.append(iv)
    
    # BS always gives flat IV = sigma
    iv_bs_flat.append(sigma)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(strikes, iv_merton, 'b-o', markersize=4, linewidth=2, label='Jump-Diffusion IV')
ax.axhline(sigma, color='r', linestyle='--', linewidth=2, label=f'BS flat IV = {sigma}')
ax.axvline(S0, color='gray', linestyle=':', alpha=0.5, label=f'$S_0$ = {S0}')
ax.set_xlabel('Strike $K$')
ax.set_ylabel('Implied Volatility')
ax.set_title(r'Volatility Smile from Merton Jump-Diffusion ($\lambda=3, \mu_J=-0.05, \sigma_J=0.1$)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("The smile is asymmetric (a 'skew') because mu_J < 0 makes downside")
print("jumps more likely, increasing OTM put prices and their implied vol.")

## Step 7: Effect of Jump Parameters on the Smile

Let's see how the jump parameters $\lambda$, $\mu_J$, and $\sigma_J$ shape the implied volatility surface.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# (a) Vary lambda (jump intensity)
for lam_i in [0, 1, 3, 5, 10]:
    ivs = []
    for Ki in strikes:
        if Ki < S0:
            p = merton_put(S0, Ki, r, T, sigma, lam_i, mu_J, sigma_J)
            ivs.append(implied_vol_newton(p, S0, Ki, r, T, is_call=False))
        else:
            p = merton_call(S0, Ki, r, T, sigma, lam_i, mu_J, sigma_J)
            ivs.append(implied_vol_newton(p, S0, Ki, r, T, is_call=True))
    label = 'BS' if lam_i == 0 else rf'$\lambda={lam_i}$'
    axes[0].plot(strikes, ivs, label=label, linewidth=1.5)
axes[0].set_title(r'Vary $\lambda$ (jump intensity)')
axes[0].set_xlabel('Strike')
axes[0].set_ylabel('Implied Vol')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# (b) Vary mu_J (mean jump size)
for mj in [0.0, -0.02, -0.05, -0.10, -0.20]:
    ivs = []
    for Ki in strikes:
        if Ki < S0:
            p = merton_put(S0, Ki, r, T, sigma, 3, mj, sigma_J)
            ivs.append(implied_vol_newton(p, S0, Ki, r, T, is_call=False))
        else:
            p = merton_call(S0, Ki, r, T, sigma, 3, mj, sigma_J)
            ivs.append(implied_vol_newton(p, S0, Ki, r, T, is_call=True))
    axes[1].plot(strikes, ivs, label=rf'$\mu_J={mj}$', linewidth=1.5)
axes[1].set_title(r'Vary $\mu_J$ (mean jump size)')
axes[1].set_xlabel('Strike')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

# (c) Vary sigma_J (jump size volatility)
for sj in [0.01, 0.05, 0.10, 0.15, 0.25]:
    ivs = []
    for Ki in strikes:
        if Ki < S0:
            p = merton_put(S0, Ki, r, T, sigma, 3, mu_J, sj)
            ivs.append(implied_vol_newton(p, S0, Ki, r, T, is_call=False))
        else:
            p = merton_call(S0, Ki, r, T, sigma, 3, mu_J, sj)
            ivs.append(implied_vol_newton(p, S0, Ki, r, T, is_call=True))
    axes[2].plot(strikes, ivs, label=rf'$\sigma_J={sj}$', linewidth=1.5)
axes[2].set_title(r'Vary $\sigma_J$ (jump volatility)')
axes[2].set_xlabel('Strike')
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

| Parameter | Effect on Smile |
|-----------|----------------|
| $\lambda$ (jump intensity) | Higher $\lambda$ amplifies the smile — more jumps = more non-Gaussian behavior |
| $\mu_J$ (mean jump size) | $\mu_J < 0$ creates a **skew** (left tail heavier); $\mu_J = 0$ gives a symmetric smile |
| $\sigma_J$ (jump volatility) | Higher $\sigma_J$ widens the smile — more variable jumps fatten both tails |
| $\lambda = 0$ | Recovers flat BS implied volatility (no jumps) |

Key takeaways:
- The Merton model naturally produces the **volatility smile/skew** observed in real markets.
- With $\mu_J < 0$ (crash-like jumps), OTM puts become more expensive — consistent with market crash protection demand.
- The MC simulation matches the analytical Merton formula, validating both implementations.
- Jump-diffusion prices are always $\geq$ BS prices — jumps add risk that the market prices in.